In [1]:
import os
import json
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, col, trim, lower, count, lit
from sentence_transformers import SentenceTransformer
from sklearn.cluster import HDBSCAN
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [2]:
def smart_capitalize(s):
    if not isinstance(s, str) or not s.strip():
        return s

    words = s.split()
    result = []

    for w in words:
        if w.isupper() and len(w) >= 2:
            result.append(w)
        elif any(c.isdigit() for c in w) or (not w.islower() and not w.isupper()):
            result.append(w)
        else:
            result.append(w.lower())

    joined = " ".join(result)
    return joined[0].upper() + joined[1:] if joined else joined

# Test nhanh
tests = ["làm việc nhóm", "SQL", "python", "AutoCAD", "microsoft office", "tiếng anh", "API gateway"]
for t in tests:
    print(f"  {t!r:30s} → {smart_capitalize(t)!r}")

  'làm việc nhóm'                → 'Làm việc nhóm'
  'SQL'                          → 'SQL'
  'python'                       → 'Python'
  'AutoCAD'                      → 'AutoCAD'
  'microsoft office'             → 'Microsoft office'
  'tiếng anh'                    → 'Tiếng anh'
  'API gateway'                  → 'API gateway'


In [3]:
spark = SparkSession.builder \
    .appName("skill_canonical_v6") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.warehouse", "s3a://warehouse") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password") \
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.nessie.authentication.type", "NONE") \
    .config("spark.sql.catalog.nessie.ref", "test") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()

print("Spark ready.")

KeyboardInterrupt: 

In [ ]:
spark.sql("""
    CALL nessie.system.expire_snapshots(
        table => 'nessie.silver.jobs',
        older_than => TIMESTAMP '2026-05-24 00:00:00',
        retain_last => 5
    )
""")

spark.sql("""
    CALL nessie.system.rewrite_data_files(
        table => 'nessie.silver.jobs',
        options => map('target-file-size-bytes', '134217728')
    )
""")

In [ ]:
rule_path = "/home/jovyan/notebooks/skill_alias_bootstrap.json"
with open(rule_path, "r", encoding="utf-8") as f:
    rule_mapping_raw = json.load(f)

rule_mapping = {}
for key, info in rule_mapping_raw.items():
    rule_mapping[key.strip().lower()] = info["canonical"]

print(f"Loaded {len(rule_mapping)} rule entries.")


In [ ]:
df_silver = spark.table("nessie.silver.jobs")

tech_df = df_silver.select(explode("tech_skills").alias("raw_skill")) \
    .filter(col("raw_skill").isNotNull() & (trim(col("raw_skill")) != "")) \
    .withColumn("skill_type", lit("tech"))

soft_df = df_silver.select(explode("soft_skills").alias("raw_skill")) \
    .filter(col("raw_skill").isNotNull() & (trim(col("raw_skill")) != "")) \
    .withColumn("skill_type", lit("soft"))

all_skills = tech_df.union(soft_df) \
    .withColumn("raw_norm", trim(lower(col("raw_skill")))) \
    .groupBy("raw_norm", "skill_type") \
    .agg(count("*").alias("freq")) \
    .orderBy(col("freq").desc())

skills_pd = all_skills.toPandas()
skills_pd["composite"] = skills_pd["raw_norm"] + "|" + skills_pd["skill_type"]
print(f"Total (raw,type) pairs: {len(skills_pd)}")
print(skills_pd["skill_type"].value_counts())

In [ ]:
mapped_composites = set(rule_mapping.keys())
skills_pd["in_rule"] = skills_pd["composite"].isin(mapped_composites)

MIN_FREQ = 3
unmapped_df = skills_pd[~skills_pd["in_rule"] & (skills_pd["freq"] >= MIN_FREQ)].copy()
print(f"Mapped by rule: {skills_pd['in_rule'].sum()}")
print(f"Unmapped with freq>={MIN_FREQ}: {len(unmapped_df)}")

In [ ]:
print("Loading multilingual-e5-large-instruct model...")
model = SentenceTransformer("intfloat/multilingual-e5-large-instruct")
model.max_seq_length = 512
print("Model loaded. Embedding dimension:", model.get_sentence_embedding_dimension())

In [ ]:
tech_canonicals = []
soft_canonicals = []
for comp, canon in rule_mapping.items():
    typ = comp.rsplit("|", 1)[1]
    if typ == "tech":
        tech_canonicals.append(canon)
    else:
        soft_canonicals.append(canon)
tech_canonicals = list(set(tech_canonicals))
soft_canonicals = list(set(soft_canonicals))
print(f"Tech canonicals: {len(tech_canonicals)}, Soft canonicals: {len(soft_canonicals)}")

In [ ]:
INSTRUCTION = "Instruct: Given a skill name, retrieve the most semantically similar standardized skill category\nQuery: "

# Embed canonical pools với instruction
print("Encoding canonical pools...")
tech_canon_emb = model.encode([INSTRUCTION + c for c in tech_canonicals], normalize_embeddings=True, batch_size=64)
soft_canon_emb = model.encode([INSTRUCTION + c for c in soft_canonicals], normalize_embeddings=True, batch_size=64)


In [ ]:
SIM_THRESHOLD = 0.93
similarity_mapping = {}

def match_batch(skills, skill_emb, canon_emb, canon_list, skill_type, results):
    BATCH = 256
    for i in range(0, len(skills), BATCH):
        batch = skill_emb[i:i+BATCH]
        sims = batch @ canon_emb.T
        best_idx = sims.argmax(axis=1)
        best_sim = sims.max(axis=1)
        for skill, idx, sim in zip(skills[i:i+BATCH], best_idx, best_sim):
            if sim >= SIM_THRESHOLD:
                comp = f"{skill}|{skill_type}"
                results[comp] = canon_list[idx]
    return results

if len(unmapped_df) > 0:
    unmapped_tech = unmapped_df[unmapped_df["skill_type"] == "tech"]["raw_norm"].tolist()
    unmapped_soft = unmapped_df[unmapped_df["skill_type"] == "soft"]["raw_norm"].tolist()
    
    if unmapped_tech:
        print(f"Encoding {len(unmapped_tech)} unmapped tech skills...")
        tech_emb = model.encode([INSTRUCTION + s for s in unmapped_tech], normalize_embeddings=True, batch_size=128, show_progress_bar=True)
        similarity_mapping = match_batch(unmapped_tech, tech_emb, tech_canon_emb, tech_canonicals, "tech", similarity_mapping)
    if unmapped_soft:
        print(f"Encoding {len(unmapped_soft)} unmapped soft skills...")
        soft_emb = model.encode([INSTRUCTION + s for s in unmapped_soft], normalize_embeddings=True, batch_size=128, show_progress_bar=True)
        similarity_mapping = match_batch(unmapped_soft, soft_emb, soft_canon_emb, soft_canonicals, "soft", similarity_mapping)
    
    print(f"Matched by similarity: {len(similarity_mapping)}")
else:
    similarity_mapping = {}

In [ ]:
matched_composites = set(similarity_mapping.keys())
remaining_df = unmapped_df[~unmapped_df["composite"].isin(matched_composites)].copy()
print(f"Remaining for clustering: {len(remaining_df)}")

def cluster_remaining(df, skill_type, model, instruction):
    if df.empty:
        return {}
    skills = df["raw_norm"].tolist()
    print(f"Clustering {len(skills)} {skill_type} skills...")
    emb = model.encode(skills, normalize_embeddings=True, batch_size=128, show_progress_bar=True)
    clusterer = HDBSCAN(min_cluster_size=2, metric='cosine')
    labels = clusterer.fit_predict(emb)
    clusters = defaultdict(list)
    for skill, label in zip(skills, labels):
        if label != -1:
            clusters[label].append(skill)
    mapping = {}
    freq_dict = df.set_index("raw_norm")["freq"].to_dict()
    for members in clusters.values():
        if len(members) >= 2:
            canonical = max(members, key=lambda x: freq_dict.get(x, 0))
            types = [df[df["raw_norm"] == m]["skill_type"].iloc[0] for m in members]
            final_type = types[0] if len(set(types)) == 1 else "mixed"
            for skill in members:
                mapping[f"{skill}|{skill_type}"] = {
                    "canonical": canonical,
                    "type": final_type
                }
    return mapping

cluster_mapping = {}
if len(remaining_df) > 0:
    tech_remain = remaining_df[remaining_df["skill_type"] == "tech"]
    if not tech_remain.empty:
        cluster_mapping.update(cluster_remaining(tech_remain, "tech", model, INSTRUCTION))
    soft_remain = remaining_df[remaining_df["skill_type"] == "soft"]
    if not soft_remain.empty:
        cluster_mapping.update(cluster_remaining(soft_remain, "soft", model, INSTRUCTION))
    print(f"Clustered: {len(cluster_mapping)}")


In [ ]:
def resolve_type(raw_norm):
    tech_exists = len(skills_pd[(skills_pd["raw_norm"] == raw_norm) & (skills_pd["skill_type"] == "tech")]) > 0
    soft_exists = len(skills_pd[(skills_pd["raw_norm"] == raw_norm) & (skills_pd["skill_type"] == "soft")]) > 0
    if tech_exists and soft_exists:
        return "tech"   # ưu tiên tech
    elif tech_exists:
        return "tech"
    elif soft_exists:
        return "soft"
    else:
        return "unknown"

final_mapping = {}
for _, row in skills_pd.iterrows():
    comp = row["composite"]
    raw_norm = row["raw_norm"]
    orig_type = row["skill_type"]
    
    if comp in rule_mapping:
        final_mapping[comp] = (rule_mapping[comp], orig_type, "rule")
    elif comp in similarity_mapping:
        final_mapping[comp] = (similarity_mapping[comp], orig_type, "sim")
    elif comp in cluster_mapping:
        final_mapping[comp] = (cluster_mapping[comp]["canonical"], cluster_mapping[comp]["type"], "cluster")
    else:
        assigned_type = resolve_type(raw_norm)
        final_mapping[comp] = (raw_norm, assigned_type, "none")


In [ ]:
rows = []
for comp, (canon, typ, src) in final_mapping.items():
    raw = comp.rsplit("|", 1)[0]
    rows.append({
        "raw_skill": raw,
        "canonical_skill": canon,
        "skill_type": typ,
        "source": src
    })
df_result = pd.DataFrame(rows)

print("\nSource distribution:")
print(df_result["source"].value_counts())
print("\nType distribution:")
print(df_result["skill_type"].value_counts())
print(f"Unknown type count: {(df_result['skill_type'] == 'unknown').sum()}")

# Lưu CSV
csv_path = "/home/jovyan/notebooks/skill_mapping_v6.csv"
df_result.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"CSV saved to {csv_path}")

# Xuất JSON
json_dict = {}
for _, row in df_result.iterrows():
    composite = f"{row['raw_skill']}|{row['skill_type']}"
    json_dict[composite] = {
        "canonical": smart_capitalize(str(row["canonical_skill"])),
        "type": row["skill_type"]
    }

json_path = "/home/jovyan/notebooks/skill_alias_v6.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(json_dict, f, ensure_ascii=False, indent=2)

print(f"JSON saved: {len(json_dict)} entries")
print(f"\nSample:")
for k, v in list(json_dict.items())[:8]:
    print(f"  {k!r:40s} → {v}")

# spark.stop()

In [ ]:
 spark.stop()

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("silver_maintenance") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.ref", "demo") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.nessie.warehouse", "s3://warehouse") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password") \
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.driver.extraJavaOptions", "-Daws.region=us-east-1") \
    .config("spark.executor.extraJavaOptions", "-Daws.region=us-east-1") \
    .getOrCreate()




In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("silver_maintenance_all_branches") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.nessie", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.nessie.uri", "http://nessie:19120/api/v1") \
    .config("spark.sql.catalog.nessie.catalog-impl", "org.apache.iceberg.nessie.NessieCatalog") \
    .config("spark.sql.catalog.nessie.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.nessie.warehouse", "s3://warehouse") \
    .config("spark.sql.catalog.nessie.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.nessie.s3.access-key-id", "admin") \
    .config("spark.sql.catalog.nessie.s3.secret-access-key", "password") \
    .config("spark.sql.catalog.nessie.s3.path-style-access", "true") \
    .config("spark.sql.catalog.nessie.cache-enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.access.key", "admin") \
    .config("spark.hadoop.fs.s3a.secret.key", "password") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.driver.extraJavaOptions", "-Daws.region=us-east-1") \
    .getOrCreate()

BRANCHES = ["demo", "test", "main"]

for branch in ["demo", "test", "main"]:
    spark.conf.set("spark.sql.catalog.nessie.ref", branch)
    
    try:
        # Bước 1: Đổi MOR (an toàn)
        spark.sql("""
            ALTER TABLE nessie.silver.jobs
            SET TBLPROPERTIES (
                'write.update.mode' = 'merge-on-read',
                'write.delete.mode' = 'merge-on-read',
                'write.merge.mode'  = 'merge-on-read'
            )
        """)
        print(f"✅ [{branch}] MOR OK")
        
        # Bước 2: Compact files (an toàn, không xóa gì)
        result = spark.sql("""
            CALL nessie.system.rewrite_data_files(
                table   => 'nessie.silver.jobs',
                options => map('target-file-size-bytes', '134217728')
            )
        """)
        row = result.collect()[0]
        print(f"✅ [{branch}] Compact: {row['rewritten_data_files_count']} → {row['added_data_files_count']} files")
        
    except Exception as e:
        print(f"⚠️  [{branch}] Bỏ qua: {e}")
        continue
    
    # Bước 1: Đổi sang Merge-on-Read
    spark.sql("""
        ALTER TABLE nessie.silver.jobs
        SET TBLPROPERTIES (
            'write.update.mode' = 'merge-on-read',
            'write.delete.mode' = 'merge-on-read',
            'write.merge.mode'  = 'merge-on-read'
        )
    """)
    print(f"  ✅ Đã đổi sang MOR trên branch '{branch}'")
    
    # Bước 2: Expire snapshots cũ
    result = spark.sql("""
        CALL nessie.system.expire_snapshots(
            table        => 'nessie.silver.jobs',
            older_than   => TIMESTAMP '2026-05-24 00:00:00',
            retain_last  => 5
        )
    """)
    row = result.collect()[0]
    print(f"  ✅ Expire xong: xóa {row['deleted_data_files_count']} data files, "
          f"{row['deleted_manifest_files_count']} manifest files")
    
    # Bước 3: Compact files
    result = spark.sql("""
        CALL nessie.system.rewrite_data_files(
            table   => 'nessie.silver.jobs',
            options => map('target-file-size-bytes', '134217728')
        )
    """)
    row = result.collect()[0]
    print(f"  ✅ Compact xong: {row['rewritten_data_files_count']} files → "
          f"{row['added_data_files_count']} files mới")

print("\n🎉 Maintenance hoàn tất trên tất cả branches!")
spark.stop()

✅ [demo] MOR OK
✅ [demo] Compact: 0 → 0 files
  ✅ Đã đổi sang MOR trên branch 'demo'


Py4JJavaError: An error occurred while calling o68.sql.
: org.apache.iceberg.exceptions.ValidationException: Cannot expire snapshots: GC is disabled (deleting files may corrupt other tables)
	at org.apache.iceberg.exceptions.ValidationException.check(ValidationException.java:49)
	at org.apache.iceberg.spark.actions.ExpireSnapshotsSparkAction.<init>(ExpireSnapshotsSparkAction.java:88)
	at org.apache.iceberg.spark.actions.SparkActions.expireSnapshots(SparkActions.java:87)
	at org.apache.iceberg.spark.procedures.ExpireSnapshotsProcedure.lambda$call$0(ExpireSnapshotsProcedure.java:116)
	at org.apache.iceberg.spark.procedures.BaseProcedure.execute(BaseProcedure.java:107)
	at org.apache.iceberg.spark.procedures.BaseProcedure.modifyIcebergTable(BaseProcedure.java:88)
	at org.apache.iceberg.spark.procedures.ExpireSnapshotsProcedure.call(ExpireSnapshotsProcedure.java:113)
	at org.apache.spark.sql.execution.datasources.v2.CallExec.run(CallExec.scala:34)
	at org.apache.spark.sql.execution.datasou